# Multi-Agent Pipeline with State Handoffs

## Introduction

The orchestrator-workers pattern is powerful when you need parallel perspectives on a single task. But what happens when agents need to work *sequentially* — where the work must first be planned, then implemented, then verified before it can ship?

This is the **multi-agent pipeline** pattern. Three specialized agents collaborate in sequence:

1. A **Planner** breaks a high-level goal into concrete, ordered subtasks
2. A **Worker** implements each subtask
3. A **Reviewer** acts as a quality gate — approving the work or sending it back with specific feedback

Without this pattern, you end up with a single agent trying to plan, implement, and verify all at once — losing the specialization benefits — or manual handoffs that break automation.

### What You'll Build

A pipeline that takes a high-level goal and:

1. Uses a **Planner agent** to decompose the goal into 3–5 ordered subtasks
2. Runs each subtask through a **Worker agent** that produces an implementation
3. Passes each result to a **Reviewer agent** that either approves or returns targeted feedback
4. Auto-retries failed subtasks with reviewer feedback injected into the worker's next attempt
5. Tracks full attempt history and fails gracefully when `max_attempts` is exceeded

### Prerequisites

- Python 3.9 or higher
- Anthropic API key set as environment variable: `export ANTHROPIC_API_KEY='your-key'`
- Basic understanding of prompt engineering
- Familiarity with Python dataclasses and enums

### When to use this pattern

**Use this pattern when:**

- A goal is too large for a single prompt but can be broken into ordered subtasks
- Work is inherently sequential — each step depends on the previous one
- Quality gates matter — you need a second agent to verify before accepting output
- Iteration improves results — reviewer feedback makes the worker's next attempt measurably better

**Don't use this pattern when:**

- Subtasks are independent and can run in parallel (use orchestrator-workers instead)
- The goal is simple enough for a single agent in one pass
- Latency is critical — each subtask costs at least two LLM calls (worker + reviewer)

## How It Works

The pipeline runs in two phases:

**Phase 1 — Planning:** The planner receives the high-level goal and returns an ordered list of concrete subtasks.

**Phase 2 — Execution:** Each subtask moves through a state machine independently:

```
PENDING → IN_PROGRESS → REVIEW → DONE
                    ↑         |
                    └─────────┘  (NEEDS_REVISION → back to IN_PROGRESS)
                                 (max attempts exceeded → FAILED)
```

**State transitions:**
- `PENDING` → `IN_PROGRESS`: worker picks up the subtask
- `IN_PROGRESS` → `REVIEW`: worker submits output
- `REVIEW` → `DONE`: reviewer approves
- `REVIEW` → `NEEDS_REVISION`: reviewer requests changes
- `NEEDS_REVISION` → `IN_PROGRESS`: worker retries with feedback
- `IN_PROGRESS` → `FAILED`: max attempts exceeded

The `Task` object carries full context across transitions — previous output, reviewer feedback, and attempt history — so each retry is informed by everything that came before.

## Setup

### Installation
```bash
pip install anthropic
```

### Helper Functions
This example uses helper functions from `util.py`:

- `llm_call(prompt, system_prompt="", model="claude-sonnet-4-6")`: Sends a prompt to Claude and returns the text response
- `extract_xml(text, tag)`: Extracts content from XML tags using regex

You can view the complete implementation in [util.py](util.py).

In [1]:
import re
from dataclasses import dataclass, field
from enum import Enum

from util import extract_xml, llm_call

MODEL = "claude-sonnet-4-6"


class TaskState(str, Enum):
    PENDING = "pending"
    IN_PROGRESS = "in_progress"
    REVIEW = "review"
    NEEDS_REVISION = "needs_revision"
    DONE = "done"
    FAILED = "failed"


@dataclass
class Task:
    """Carries subtask context across all state transitions."""

    description: str
    state: TaskState = TaskState.PENDING
    result: str = ""
    feedback: str = ""
    attempts: int = 0
    history: list = field(default_factory=list)

## Implementation

The `MultiAgentPipeline` class coordinates all three agents:

- `_planner()`: Receives the high-level goal and returns an ordered list of subtask strings. Uses `re.findall` to parse multiple `<subtask>` tags from the response.
- `_worker()`: Receives a subtask description plus any reviewer feedback from prior attempts. Returns an implementation wrapped in `<result>` tags.
- `_reviewer()`: Receives both the original subtask description and the worker's output. Returns a `<verdict>` (`APPROVE` or `REVISE`) and `<feedback>` explaining the decision.

**Key design decisions:**
- The planner always sees the full goal, not individual subtasks — this lets it reason about ordering and dependencies upfront
- The `Task` dataclass is the single source of truth per subtask — it accumulates result, feedback, and history across retries so nothing is lost between state transitions
- The reviewer always sees the *original subtask description*, not just the output — this prevents drift where a reviewer grades against what was built rather than what was asked
- Reviewer feedback is injected verbatim into the worker's retry prompt — the worker gets the exact notes, not a summary
- `max_attempts` is enforced at the `REVIEW` → `NEEDS_REVISION` transition, so the final attempt always gets reviewed before the task is marked `FAILED`

In [2]:
class MultiAgentPipeline:
    """Three-agent pipeline: planner decomposes a goal, worker implements each subtask,
    reviewer acts as a quality gate with auto-retry on revision requests."""

    def __init__(
        self,
        planner_prompt: str,
        worker_prompt: str,
        reviewer_prompt: str,
        max_attempts: int = 3,
        model: str = MODEL,
    ):
        self.planner_prompt = planner_prompt
        self.worker_prompt = worker_prompt
        self.reviewer_prompt = reviewer_prompt
        self.max_attempts = max_attempts
        self.model = model

    def _planner(self, goal: str) -> list[str]:
        """Planner agent: decomposes a high-level goal into ordered subtasks."""
        prompt = self.planner_prompt.format(goal=goal)
        response = llm_call(prompt, model=self.model)
        subtasks_block = extract_xml(response, "subtasks")
        subtasks = re.findall(r"<subtask>(.*?)</subtask>", subtasks_block, re.DOTALL)
        return [s.strip() for s in subtasks if s.strip()]

    def _worker(self, task: Task) -> str:
        """Worker agent: implements a subtask, incorporating reviewer feedback if present."""
        context = ""
        if task.feedback:
            context = (
                f"\n\nYour previous attempt:\n{task.result}"
                f"\n\nReviewer feedback:\n{task.feedback}"
                f"\n\nAddress the feedback above in your revised implementation."
            )

        prompt = self.worker_prompt.format(task=task.description, context=context)
        response = llm_call(prompt, model=self.model)
        return extract_xml(response, "result")

    def _reviewer(self, task: Task) -> tuple[str, str]:
        """Reviewer agent: evaluates worker output against the original subtask."""
        prompt = self.reviewer_prompt.format(
            task=task.description,
            result=task.result,
        )
        response = llm_call(prompt, model=self.model)
        verdict = extract_xml(response, "verdict").strip().upper()
        feedback = extract_xml(response, "feedback").strip()
        return verdict, feedback

    def _run_subtask(self, task: Task) -> Task:
        """Run a single subtask through the worker → reviewer state machine."""
        while task.state not in (TaskState.DONE, TaskState.FAILED):
            if task.state in (TaskState.PENDING, TaskState.NEEDS_REVISION):
                task.state = TaskState.IN_PROGRESS
                task.attempts += 1

                print(f"  [Attempt {task.attempts}/{self.max_attempts}] Worker implementing...")

                result = self._worker(task)
                task.result = result or "[Worker produced no output]"
                task.history.append({"attempt": task.attempts, "result": task.result})
                task.state = TaskState.REVIEW

                print(f"\n  Worker output:\n{task.result}")

            elif task.state == TaskState.REVIEW:
                print(f"\n  [Attempt {task.attempts}] Reviewer evaluating...")

                verdict, feedback = self._reviewer(task)

                print(f"  Verdict: {verdict}")
                if feedback:
                    print(f"  Feedback: {feedback}")

                if verdict == "APPROVE":
                    task.state = TaskState.DONE
                    task.feedback = ""
                elif task.attempts >= self.max_attempts:
                    task.state = TaskState.FAILED
                    task.feedback = feedback
                    print(f"\n  [!] Max attempts ({self.max_attempts}) reached. Marking as FAILED.")
                else:
                    task.state = TaskState.NEEDS_REVISION
                    task.feedback = feedback

        return task

    def run(self, goal: str) -> list[Task]:
        """Plan the goal into subtasks, then run each through the worker-reviewer loop."""
        print(f"\n{'=' * 70}")
        print(f"GOAL: {goal}")
        print(f"{'=' * 70}")

        # Phase 1: Plan
        print("\n[Planner] Decomposing goal into subtasks...")
        subtasks = self._planner(goal)

        if not subtasks:
            print("[!] Planner returned no subtasks. Aborting.")
            return []

        print(f"\nPlan ({len(subtasks)} subtasks):")
        for i, s in enumerate(subtasks, 1):
            print(f"  {i}. {s}")

        # Phase 2: Execute
        completed: list[Task] = []
        for i, subtask in enumerate(subtasks, 1):
            print(f"\n{'─' * 70}")
            print(f"Subtask {i}/{len(subtasks)}: {subtask}")
            print(f"{'─' * 70}\n")

            task = Task(description=subtask)
            completed.append(self._run_subtask(task))

        # Summary
        done = sum(1 for t in completed if t.state == TaskState.DONE)
        failed = sum(1 for t in completed if t.state == TaskState.FAILED)

        print(f"\n{'=' * 70}")
        print(f"PIPELINE COMPLETE — {done} done, {failed} failed out of {len(completed)} subtasks")
        print(f"{'=' * 70}")

        return completed

## Example Use Case: Building a Roman Numeral Module

Let's give the pipeline a high-level goal and watch the planner decompose it, then the worker and reviewer handle each piece.

**Why this example demonstrates the pattern well:**
- The goal is clearly decomposable — encode, decode, validate, and test are natural subtasks with a logical order
- Each subtask is independently reviewable — the reviewer has a clear bar for each piece
- The planner's decomposition is non-trivial — it must reason about dependencies (validation logic should exist before tests exercise it)
- The reviewer can provide targeted, actionable feedback per subtask rather than a vague overall assessment

**Prompt design notes:**
- The planner prompt asks for 3–5 subtasks and requires logical ordering — this prevents it from generating subtasks that depend on work not yet done
- The worker prompt has a `{context}` slot that is empty on the first attempt and filled with the reviewer's exact feedback on retries — same template, progressively richer input
- The reviewer prompt always includes the *original subtask description* alongside the output — prevents grade drift
- All prompts use XML output tags for reliable parsing

In [3]:
PLANNER_PROMPT = """You are a planning agent. Break the following goal into concrete, ordered subtasks.

Goal:
{goal}

Requirements:
- Each subtask must be independently implementable and reviewable
- Order subtasks so dependencies come first
- Keep each subtask focused — one clear deliverable
- Aim for 3–5 subtasks

Return your plan in this format:

<subtasks>
  <subtask>Description of subtask 1</subtask>
  <subtask>Description of subtask 2</subtask>
</subtasks>
"""

WORKER_PROMPT = """You are a software engineer. Implement the following subtask.{context}

Subtask:
{task}

Return your implementation in this format:

<result>
Your complete implementation here, including all code, docstrings, and type hints.
</result>
"""

REVIEWER_PROMPT = """You are a senior code reviewer. Evaluate the implementation below against the subtask requirements.

Subtask:
{task}

Implementation:
{result}

Review for:
1. Correctness — does it fully solve the subtask as specified?
2. Edge cases — are obvious edge cases handled?
3. Clarity — is the code readable, well-typed, and documented?

Return your review in this format:

<verdict>APPROVE or REVISE</verdict>
<feedback>
If approving: briefly explain why it meets the bar.
If requesting revision: list exactly what needs to change, one item per line.
</feedback>
"""

pipeline = MultiAgentPipeline(
    planner_prompt=PLANNER_PROMPT,
    worker_prompt=WORKER_PROMPT,
    reviewer_prompt=REVIEWER_PROMPT,
    max_attempts=3,
)

results = pipeline.run(
    "Build a Python module for Roman numeral conversion. "
    "It should convert integers to Roman numerals and Roman numerals to integers, "
    "validate input, handle edge cases, and include a test suite."
)


GOAL: Build a Python module for Roman numeral conversion. It should convert integers to Roman numerals and Roman numerals to integers, validate input, handle edge cases, and include a test suite.

[Planner] Decomposing goal into subtasks...



Plan (4 subtasks):
  1. Define the core data structures and constants for Roman numeral conversion: create a Python module file (`roman.py`) that establishes the ordered mapping of integer values to Roman numeral symbols (e.g., a list of `(value, symbol)` tuples covering all standard numerals including subtractive forms like IV, IX, XL, etc.).
  2. Implement the `int_to_roman(n)` function with input validation: accept a positive integer, raise a `ValueError` for out-of-range inputs (e.g., less than 1 or greater than 3999), and use the value-symbol mapping to iteratively build and return the correct Roman numeral string.
  3. Implement the `roman_to_int(s)` function with input validation: accept a Roman numeral string, raise a `ValueError` for invalid or empty input, and parse the string left-to-right using subtractive logic (if a symbol is less than the next symbol, subtract it; otherwise add it) to return the corresponding integer.
  4. Write a comprehensive test suite (`test_roman.p


  Worker output:

"""
roman.py - Core data structures and constants for Roman numeral conversion.

This module defines the foundational mapping between integer values and
Roman numeral symbols, including all standard subtractive forms.
"""

from typing import List, Tuple

# Ordered mapping of integer values to Roman numeral symbols.
# The list is ordered from largest to smallest value to facilitate
# greedy conversion algorithms. Includes all standard subtractive forms
# (IV, IX, XL, XC, CD, CM) as defined by classical Roman numeral rules.
ROMAN_NUMERAL_MAP: List[Tuple[int, str]] = [
    (1000, "M"),
    (900,  "CM"),
    (500,  "D"),
    (400,  "CD"),
    (100,  "C"),
    (90,   "XC"),
    (50,   "L"),
    (40,   "XL"),
    (10,   "X"),
    (9,    "IX"),
    (5,    "V"),
    (4,    "IV"),
    (1,    "I"),
]

# Minimum and maximum integer values supported by standard Roman numerals
MIN_VALUE: int = 1
MAX_VALUE: int = 3999

# Individual Roman numeral symbols and their base values (for 

  Verdict: APPROVE
  Feedback: The implementation fully meets the subtask requirements and exceeds the minimum bar in several ways:

1. **Correctness**: `ROMAN_NUMERAL_MAP` contains all 13 standard entries (7 additive + 6 subtractive forms: IV, IX, XL, XC, CD, CM), ordered largest-to-smallest, which is exactly what a greedy conversion algorithm needs.

2. **Edge cases**: `MIN_VALUE` and `MAX_VALUE` constants (1 and 3999) are defined, signaling awareness of the valid input range for standard Roman numerals. These will be useful for validation in downstream conversion functions.

3. **Clarity**: The module is well-documented with a module-level docstring, inline comments explaining the ordering rationale and subtractive forms, and proper type annotations (`List[Tuple[int, str]]`). The naming is clear and conventional.

4. **Extras**: `ROMAN_SYMBOLS` and `ROMAN_VALUES` derived lists are a reasonable addition for reference/utility, though they are not strictly required by the subtask. They


  Worker output:

def int_to_roman(n: int) -> str:
    """
    Convert a positive integer to its Roman numeral representation.

    Args:
        n: A positive integer between 1 and 3999 (inclusive).

    Returns:
        A string containing the Roman numeral representation of n.

    Raises:
        ValueError: If n is less than 1 or greater than 3999.
        TypeError: If n is not an integer.

    Examples:
        >>> int_to_roman(1)
        'I'
        >>> int_to_roman(4)
        'IV'
        >>> int_to_roman(9)
        'IX'
        >>> int_to_roman(58)
        'LVIII'
        >>> int_to_roman(1994)
        'MCMXCIV'
        >>> int_to_roman(3999)
        'MMMCMXCIX'
    """
    if not isinstance(n, int) or isinstance(n, bool):
        raise TypeError(f"Expected an integer, got {type(n).__name__}")

    if n < 1 or n > 3999:
        raise ValueError(f"Input must be between 1 and 3999, got {n}")

    value_symbol_mapping = [
        (1000, "M"),
        (900, "CM"),
        (500, 

...

  Verdict: APPROVE
  Feedback: The implementation correctly satisfies all subtask requirements:

1. **Correctness**: The left-to-right subtractive logic is properly implemented — when a symbol's value is less than the next symbol's value, it's subtracted; otherwise it's added. This correctly handles cases like IV (4), IX (9), XL (40), XC (90), CD (400), CM (900), and complex combinations like MCMXCIX (1999).

2. **Input validation**: All three required validation cases are covered — non-string input raises ValueError with a descriptive message, empty string raises ValueError, and invalid characters (after uppercasing) raise ValueError. The `.upper()` normalization is a nice usability touch.

3. **Edge cases**: Single characters work correctly (the else branch handles the last character). The `prev_value` variable is declared but unused — this is a minor dead code issue but doesn't affect correctness. Mixed case input is handled gracefully via `.upper()`.

4. **Clarity**: The code is cl


  Worker output:
[Worker produced no output]

  [Attempt 1] Reviewer evaluating...


  Verdict: REVISE
  Feedback: The worker produced no output — there is no implementation to review. A complete `test_roman.py` file must be written from scratch. It should include:
- Standard conversion tests (e.g., 1→"I", 4→"IV", 9→"IX", 40→"XL", 90→"XC", 400→"CD", 900→"CM", 1994→"MCMXCIV", 3999→"MMMCMXCIX")
- Boundary value tests for both 1 and 3999 in both directions
- Explicit tests for all six subtractive pairs: IV, IX, XL, XC, CD, CM
- Invalid input tests: 0, negative integers, integers > 3999, non-integer/non-string types (float, None, list, etc.), and malformed Roman numeral strings (e.g., "IIII", "VV", "IC", "", "ABC")
- Round-trip consistency tests (int → Roman → int) for a representative range of values (e.g., 1–3999 or a sampled subset)
- Use of `unittest.TestCase` (or pytest) with clearly named test methods and appropriate assertions (`assertEqual`, `assertRaises`, etc.)
- Readable structure with logical grouping of test cases and docstrings or comments explaining intent
 


  Worker output:

"""
test_roman.py

Comprehensive test suite for a Roman numeral converter module.

Assumes the module `roman` exposes two functions:
    - to_roman(n: int) -> str   : converts an integer to a Roman numeral string
    - from_roman(s: str) -> int : converts a Roman numeral string to an integer

Both functions should raise ValueError for invalid inputs.
"""

import unittest

try:
    from roman import to_roman, from_roman
except ImportError:
    # Provide stub implementations so the test file can be imported/inspected
    # even when the real module is not yet present.
    def to_roman(n):
        raise NotImplementedError("roman.to_roman is not implemented")

    def from_roman(s):
        raise NotImplementedError("roman.from_roman is not implemented")


class TestToRomanStandardConversions(unittest.TestCase):
    """Standard integer → Roman numeral conversions."""

    def test_one(self):
        """1 should convert to 'I'."""
        self.assertEqual(to_roman(1), "I

  Verdict: APPROVE
  Feedback: This test suite comprehensively meets and exceeds the subtask requirements. Here's why it passes review:

**Correctness:**
- Covers all required areas: standard conversions in both directions, boundary values (1 and 3999), all six subtractive pairs (IV, IX, XL, XC, CD, CM), invalid inputs, and round-trip consistency.
- The full range round-trip test (1–3999) using `subTest` is thorough and well-structured.
- Invalid input tests correctly use `assertRaises` with appropriate exception types (`ValueError`, `TypeError`, or both).
- The graceful fallback stub import pattern allows the test file to be imported even without the implementation module present.

**Edge Cases:**
- Malformed Roman strings are well-covered: empty string, repeated non-repeatable symbols (VV, LL, DD), four consecutive identical symbols (IIII, MMMM), invalid subtractive pairs (IC, IL, VX), lowercase/mixed-case, whitespace, special characters, and plain numeric strings.
- Type checking co

## Summary

You've implemented a three-agent pipeline where a planner decomposes a high-level goal, a worker implements each subtask, and a reviewer acts as a quality gate. Each subtask moves through a formal state machine with auto-retry — the reviewer's feedback flows directly into the worker's next attempt, and the pipeline fails gracefully when `max_attempts` is reached.

### Key Takeaways

**Pattern benefits:**
- **Specialization**: Each agent is optimized for one job — planning, implementing, or evaluating
- **Informed retries**: Reviewer feedback is injected verbatim into the worker's retry prompt, making each attempt better than the last
- **Auditability**: `Task.history` records every attempt and its outcome — useful for debugging and understanding why a subtask failed
- **Graceful failure**: `max_attempts` prevents infinite loops; the final state of every subtask is always deterministic
- **Separation of concerns**: The planner reasons about the whole goal; the worker and reviewer focus on one subtask at a time

**When this pattern excels:**
- Code generation pipelines where a goal can be split into independently reviewable pieces
- Document drafting with planning (outline) → writing → editorial review
- Data processing workflows where each transformation step needs validation before the next begins
- Any multi-step task where a dedicated reviewer catching mistakes outweighs the cost of extra LLM calls

### Limitations & Considerations

**Cost & Latency:**
- Each subtask costs at least two LLM calls (worker + reviewer), plus one upfront planner call
- With `max_attempts=3` and 4 subtasks, worst case is 1 + (3 × 2 × 4) = 25 calls
- Use `claude-haiku-4-5` for the reviewer if your review criteria are straightforward

**Planner quality matters:**
- Vague subtasks produce vague worker output and vague reviewer feedback
- If the planner generates subtasks that are too large, consider adding a max subtask scope constraint to the planner prompt

**Reviewer prompt quality matters:**
- Be explicit about what `APPROVE` requires — the reviewer will meet the bar you set, not a higher one
- A vague reviewer prompt produces vague feedback, which produces only marginally better worker output on retry

**Failure modes to consider:**
- Reviewer and worker can get stuck in a loop if the reviewer's feedback is inconsistent between attempts
- Consider logging `Task.history` to detect loops where the worker produces nearly identical output across retries
- Subtasks generated by the planner are currently independent — if one fails, the pipeline continues with the rest rather than aborting

### Next Steps

**Extend the pipeline:**
1. Run subtasks concurrently using `asyncio` when they have no dependencies on each other
2. Add a **synthesis agent** that combines all approved subtask outputs into a final cohesive result
3. Add dependency tracking to the planner output so subtasks can declare which prior subtasks they depend on
4. Persist `Task` state to disk so the pipeline can resume after interruption in long-running workflows

**Adapt the prompts:**
- Swap the coding goal for document writing, data extraction, or test generation — the state machine is domain-agnostic
- Tune the planner prompt to control subtask granularity for your use case
- Use a stricter reviewer (higher bar for `APPROVE`) when output quality is critical, or a more lenient one when speed matters more